In [40]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: True
GPU Name: Tesla T4


In [41]:
# STEP 1: Install Dependencies
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install transformers
!pip install transformers torch
!pip install spacy
!pip install faiss-cpu
!pip install sentence-transformers
!pip install gradio
!pip install lime

# Download spaCy model
!python -m spacy download en_core_web_sm

print("All dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All dependencies installed!


In [42]:
# Create Folder Structure
import os

folders = [
    'mental_health_chatbot/backend/models',
    'mental_health_chatbot/backend/data',
    'mental_health_chatbot/backend/utils',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folder structure created!")

Folder structure created!


In [43]:
# Upload datasets

import json
import shutil
import os
from google.colab import files

os.makedirs('mental_health_chatbot/backend/data', exist_ok=True)

print("Please upload ALL 4 data files at once:")
print("   - training_data.json")
print("   - cbt_knowledge.json")
print("   - crisis_keywords.json")
print("   - mental_health_kg.json")
print()

uploaded = files.upload()

# Move all uploaded files
for filename in uploaded.keys():
    destination = f'mental_health_chatbot/backend/data/{filename}'
    shutil.move(filename, destination)
    print(f"Moved {filename} to {destination}")

print("\nAll files uploaded!")

Please upload ALL 4 data files at once:
   - training_data.json
   - cbt_knowledge.json
   - crisis_keywords.json
   - mental_health_kg.json



Saving cbt_knowledge.json to cbt_knowledge.json
Saving crisis_keywords.json to crisis_keywords.json
Saving mental_health_kg.json to mental_health_kg.json
Saving training_data.json to training_data.json
Moved cbt_knowledge.json to mental_health_chatbot/backend/data/cbt_knowledge.json
Moved crisis_keywords.json to mental_health_chatbot/backend/data/crisis_keywords.json
Moved mental_health_kg.json to mental_health_chatbot/backend/data/mental_health_kg.json
Moved training_data.json to mental_health_chatbot/backend/data/training_data.json

All files uploaded!


In [44]:
import networkx as nx
import json

with open('mental_health_chatbot/backend/data/mental_health_kg.json', 'r', encoding='utf-8') as f:
    kg_data = json.load(f)

G = nx.DiGraph()

for condition, details in kg_data.items():
    cond_id = condition.lower().replace(" ", "_")
    G.add_node(cond_id, type="condition", name=condition)

    for symptom in details.get("symptoms", []):
        sym_id = symptom.lower().replace(" ", "_")
        G.add_node(sym_id, type="symptom")
        G.add_edge(cond_id, sym_id, relation="has_symptom")

    for cause in details.get("causes", []):
        cause_id = cause.lower().replace(" ", "_")
        G.add_node(cause_id, type="cause")
        G.add_edge(cond_id, cause_id, relation="caused_by")

    for treatment in details.get("treatments", []):
        treat_id = treatment.lower().replace(" ", "_")
        G.add_node(treat_id, type="treatment")
        G.add_edge(cond_id, treat_id, relation="treated_by")

print(f"Knowledge Graph Created : {len(G.nodes)} nodes, {len(G.edges)} edges")

Knowledge Graph Created : 217 nodes, 316 edges


In [45]:
def extract_graph_nodes(text):
    keywords = [
        "stress", "insomnia", "fatigue",
        "isolation", "anxiety", "overthinking",
        "sleep_loss", "depression"
    ]

    found = []
    for word in keywords:
        if word in text.lower():
            found.append(word)

    return found

def graph_reasoning(nodes):

    expanded = set(nodes)

    for node in nodes:
        if node in G:
            neighbors = list(G.successors(node))
            expanded.update(neighbors)

    return list(expanded)

def graph_risk(expanded_nodes):

    if "depression" in expanded_nodes and "isolation" in expanded_nodes:
        return "high"

    if "fatigue" in expanded_nodes or "insomnia" in expanded_nodes:
        return "medium"

    return "low"

def explain_risk(nodes, expanded):

    return f"⚠️ Risk detected due to: {', '.join(nodes)} → {', '.join(expanded)}"

In [48]:
import torch
from torch_geometric.nn import TransformerConv

class GraphTransformer(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = TransformerConv(8, 16)
        self.conv2 = TransformerConv(16, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.conv2(x, edge_index)
        return x

In [47]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import json
import os

# Load training_data.json
data_path = 'mental_health_chatbot/backend/data/training_data.json'
try:
    with open(data_path, 'r', encoding='utf-8') as f:
        training_data = json.load(f)
    print(f"Loaded {len(training_data)} items from {data_path}")
except FileNotFoundError:
    print(f"Error: {data_path} not found. Please ensure the file is uploaded.")
    training_data = [] # Initialize as empty list to prevent further errors
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from {data_path}. Please check the file's integrity.")
    training_data = [] # Initialize as empty list

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

dataset_texts = [item["text"] for item in training_data]
dataset_responses = [item["response"] for item in training_data]

print("Encoding dataset...")
embeddings = embed_model.encode(dataset_texts)
embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS ready!")

def retrieve_response(query):
    query_vec = embed_model.encode([query]).astype("float32")
    D, I = index.search(query_vec, k=3)

    return [dataset_responses[i] for i in I[0]]

Loaded 336 items from mental_health_chatbot/backend/data/training_data.json


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding dataset...
FAISS ready!


In [49]:
# File: mental_health_chatbot/backend/models/emotion_detector.py

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

class EmotionDetector:
    def __init__(self):
        """Initialize emotion detection model"""
        print("Loading emotion detection model...")

        # Using a pre-trained emotion classification model
        model_name = "bhadresh-savani/distilbert-base-uncased-emotion"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Emotion labels
        self.emotions = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

        print("Emotion detector loaded!")

    def detect(self, text):
        """
        Detect emotion from text

        Args:
            text (str): Input text

        Returns:
            dict: Emotion probabilities and top emotion
        """
        # Tokenize
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

        # Get predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Get probabilities
        probs = predictions[0].cpu().numpy()

        # Create result dictionary
        emotion_scores = {emotion: float(prob) for emotion, prob in zip(self.emotions, probs)}

        # Get top emotion
        top_emotion = self.emotions[np.argmax(probs)]
        top_score = float(np.max(probs))

        return {
            'top_emotion': top_emotion,
            'confidence': top_score,
            'all_emotions': emotion_scores
        }

    def analyze_batch(self, texts):
        """Analyze multiple texts at once"""
        results = []
        for text in texts:
            results.append(self.detect(text))
        return results

# Save the file
emotion_code = '''import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

class EmotionDetector:
    def __init__(self):
        """Initialize emotion detection model"""
        print("Loading emotion detection model...")

        # Using a pre-trained emotion classification model
        model_name = "bhadresh-savani/distilbert-base-uncased-emotion"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Emotion labels
        self.emotions = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

        print("Emotion detector loaded!")

    def detect(self, text):
        """
        Detect emotion from text

        Args:
            text (str): Input text

        Returns:
            dict: Emotion probabilities and top emotion
        """
        # Tokenize
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

        # Get predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Get probabilities
        probs = predictions[0].cpu().numpy()

        # Create result dictionary
        emotion_scores = {emotion: float(prob) for emotion, prob in zip(self.emotions, probs)}

        # Get top emotion
        top_emotion = self.emotions[np.argmax(probs)]
        top_score = float(np.max(probs))

        return {
            'top_emotion': top_emotion,
            'confidence': top_score,
            'all_emotions': emotion_scores
        }

    def analyze_batch(self, texts):
        """Analyze multiple texts at once"""
        results = []
        for text in texts:
            results.append(self.detect(text))
        return results
'''

with open('mental_health_chatbot/backend/models/emotion_detector.py', 'w') as f:
    f.write(emotion_code)

print("emotion_detector.py created!")

emotion_detector.py created!


In [50]:
# TEST: Emotion Detection Module

# Import the module
import sys
sys.path.append('mental_health_chatbot/backend')

from models.emotion_detector import EmotionDetector

# Initialize detector
detector = EmotionDetector()

# Test cases
test_messages = [
    "I feel so lonely and sad all the time",
    "I'm really anxious about my exams",
    "I haven't been able to sleep for days",
    "I feel hopeless and don't want to live anymore",
    "Today was a great day! I'm so happy!"
]

print("\n" + "="*60)
print("TESTING EMOTION DETECTION")
print("="*60 + "\n")

for msg in test_messages:
    result = detector.detect(msg)

    print(f"Message: {msg}")
    print(f"Emotion: {result['top_emotion']} ({result['confidence']:.2%})")
    print(f"All scores: {result['all_emotions']}")
    print("-"*60 + "\n")

Loading emotion detection model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Emotion detector loaded!

TESTING EMOTION DETECTION

Message: I feel so lonely and sad all the time
Emotion: sadness (99.90%)
All scores: {'sadness': 0.9989677667617798, 'joy': 0.00017520862456876785, 'love': 0.00025895191356539726, 'anger': 0.00026751664699986577, 'fear': 0.00020677289285231382, 'surprise': 0.00012401884305290878}
------------------------------------------------------------

Message: I'm really anxious about my exams
Emotion: fear (99.69%)
All scores: {'sadness': 0.0009141279733739793, 'joy': 0.00044840440386906266, 'love': 0.00013766944175586104, 'anger': 0.0012072814861312509, 'fear': 0.9968801736831665, 'surprise': 0.000412405381212011}
------------------------------------------------------------

Message: I haven't been able to sleep for days
Emotion: joy (97.12%)
All scores: {'sadness': 0.014155631884932518, 'joy': 0.9711869955062866, 'love': 0.000944067956879735, 'anger': 0.006379193160682917, 'fear': 0.006134037859737873, 'surprise': 0.001200131606310606}
-----

In [51]:
# File: mental_health_chatbot/backend/models/intent_classifier.py

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
import os
import json

class IntentClassifier:
    def __init__(self, model_name="distilroberta-base", intents=None):
        print("Loading intent classification model...")

        if intents is None:
            self.intents = ['crisis', 'help_seeking', 'general']
        else:
            self.intents = intents

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(self.intents),
            problem_type="single_label_classification"
        )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
        self.model.to(self.device)

        self.label2id = {label: i for i, label in enumerate(self.intents)}
        self.id2label = {i: label for i, label in enumerate(self.intents)}

        print(f"Initialized with {len(self.intents)} intents: {self.intents}")

    @staticmethod
    def clean_label(text):
        """Rule-based fallback labeler"""
        text = text.lower()
        if any(k in text for k in ["kill myself", "suicide", "end my life", "want to die", "i want to end everything"]):
            return "crisis"
        if any(k in text for k in ["help", "advice", "what should", "how do", "can you", "i need help"]):
            return "help_seeking"
        if any(k in text for k in ["hello", "hi", "thanks", "how are you"]):
            return "general"
        return "general"

    def _tokenize_function(self, examples):
        return self.tokenizer(
            examples['text'],
            truncation=True,
            max_length=128,
            padding='max_length'
        )

    def _compute_metrics(self, pred):
        labels = pred.label_ids
        preds = np.argmax(pred.predictions, axis=1)
        accuracy = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')
        return {'accuracy': accuracy, 'f1': f1}

    def train(self, train_texts, train_labels, eval_texts=None, eval_labels=None, epochs=5):
        print("Starting training for IntentClassifier...")

        train_label_ids = [self.label2id[label] for label in train_labels]

        train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_label_ids})
        tokenized_train = train_dataset.map(self._tokenize_function, batched=True)

        tokenized_eval = None
        if eval_texts and eval_labels:
            eval_label_ids = [self.label2id[label] for label in eval_labels]
            eval_dataset = Dataset.from_dict({'text': eval_texts, 'labels': eval_label_ids})
            tokenized_eval = eval_dataset.map(self._tokenize_function, batched=True)

        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=epochs,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=8,
            gradient_accumulation_steps=2,
            warmup_steps=50,
            weight_decay=0.05,
            learning_rate=3e-5,
            logging_steps=5,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            report_to="none",
            save_total_limit=2,
            fp16=torch.cuda.is_available(),
            use_cpu=False,
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_eval,
            compute_metrics=self._compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

        print("Training in progress...")
        trainer.train()
        print("Training completed!")

    def classify(self, text):
        self.model.eval()
        inputs = self.tokenizer(text, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
        top_idx = np.argmax(probs)

        return {
            "intent": self.id2label[top_idx],
            "confidence": float(probs[top_idx]),
            "all_intents": {intent: float(prob) for intent, prob in zip(self.intents, probs)}
        }

    def analyze_batch(self, texts):
        """Analyze multiple texts at once"""
        results = []
        for text in texts:
            results.append(self.classify(text))
        return results

    def predict(self, text):
        return self.classify(text)

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)

        config = {
            "intents": self.intents,
            "label2id": self.label2id,
            "id2label": {str(k): v for k, v in self.id2label.items()}
        }
        with open(os.path.join(path, "labels.json"), "w") as f:
            json.dump(config, f, indent=2)
        print(f"Model saved to {path}")

    @classmethod
    def load(cls, path):
        print(f"Loading intent classifier from {path}...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(path)
            model = AutoModelForSequenceClassification.from_pretrained(path)
            with open(os.path.join(path, "labels.json")) as f:
                config = json.load(f)

            instance = cls.__new__(cls)
            instance.intents = config.get("intents", ["crisis", "help_seeking", "general"])
            instance.label2id = {label: i for i, label in enumerate(instance.intents)}
            instance.id2label = {i: label for i, label in enumerate(instance.intents)}
            instance.tokenizer = tokenizer
            instance.model = model
            instance.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            instance.model.to(instance.device)

            print("Intent classifier loaded successfully!")
            return instance
        except Exception as e:
            print(f"Error loading: {e}. Creating new model.")
            return cls()

intent_code = '''# File: mental_health_chatbot/backend/models/intent_classifier.py

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
import os
import json

class IntentClassifier:
    def __init__(self, model_name="distilroberta-base", intents=None):
        print("Loading intent classification model...")

        if intents is None:
            self.intents = ['crisis', 'help_seeking', 'general']
        else:
            self.intents = intents

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(self.intents),
            problem_type="single_label_classification"
        )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
        self.model.to(self.device)

        self.label2id = {label: i for i, label in enumerate(self.intents)}
        self.id2label = {i: label for i, label in enumerate(self.intents)}

        print(f"Initialized with {len(self.intents)} intents: {self.intents}")

    @staticmethod
    def clean_label(text):
        """Rule-based fallback labeler"""
        text = text.lower()
        if any(k in text for k in ["kill myself", "suicide", "end my life", "want to die", "i want to end everything"]):
            return "crisis"
        if any(k in text for k in ["help", "advice", "what should", "how do", "can you", "i need help"]):
            return "help_seeking"
        if any(k in text for k in ["hello", "hi", "thanks", "how are you"]):
            return "general"
        return "general"

    def _tokenize_function(self, examples):
        return self.tokenizer(
            examples['text'],
            truncation=True,
            max_length=128,
            padding='max_length'
        )

    def _compute_metrics(self, pred):
        labels = pred.label_ids
        preds = np.argmax(pred.predictions, axis=1)
        accuracy = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')
        return {'accuracy': accuracy, 'f1': f1}

    def train(self, train_texts, train_labels, eval_texts=None, eval_labels=None, epochs=5):
        print("Starting training for IntentClassifier...")

        train_label_ids = [self.label2id[label] for label in train_labels]

        train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_label_ids})
        tokenized_train = train_dataset.map(self._tokenize_function, batched=True)

        tokenized_eval = None
        if eval_texts and eval_labels:
            eval_label_ids = [self.label2id[label] for label in eval_labels]
            eval_dataset = Dataset.from_dict({'text': eval_texts, 'labels': eval_label_ids})
            tokenized_eval = eval_dataset.map(self._tokenize_function, batched=True)

        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=epochs,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=8,
            gradient_accumulation_steps=2,
            warmup_steps=50,
            weight_decay=0.05,
            learning_rate=3e-5,
            logging_steps=5,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            report_to="none",
            save_total_limit=2,
            fp16=torch.cuda.is_available(),
            use_cpu=False,
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_eval,
            compute_metrics=self._compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

        print("Training in progress...")
        trainer.train()
        print("Training completed!")

    def classify(self, text):
        self.model.eval()
        inputs = self.tokenizer(text, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
        top_idx = np.argmax(probs)

        return {
            "intent": self.id2label[top_idx],
            "confidence": float(probs[top_idx]),
            "all_intents": {intent: float(prob) for intent, prob in zip(self.intents, probs)}
        }

    def analyze_batch(self, texts):
        """Analyze multiple texts at once"""
        results = []
        for text in texts:
            results.append(self.classify(text))
        return results

    def predict(self, text):
        return self.classify(text)

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)

        config = {
            "intents": self.intents,
            "label2id": self.label2id,
            "id2label": {str(k): v for k, v in self.id2label.items()}
        }
        with open(os.path.join(path, "labels.json"), "w") as f:
            json.dump(config, f, indent=2)
        print(f"Model saved to {path}")

    @classmethod
    def load(cls, path):
        print(f"Loading intent classifier from {path}...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(path)
            model = AutoModelForSequenceClassification.from_pretrained(path)
            with open(os.path.join(path, "labels.json")) as f:
                config = json.load(f)

            instance = cls.__new__(cls)
            instance.intents = config.get("intents", ["crisis", "help_seeking", "general"])
            instance.label2id = {label: i for i, label in enumerate(instance.intents)}
            instance.id2label = {i: label for i, label in enumerate(instance.intents)}
            instance.tokenizer = tokenizer
            instance.model = model
            instance.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            instance.model.to(instance.device)

            print("Intent classifier loaded successfully!")
            return instance
        except Exception as e:
            print(f"Error loading: {e}. Creating new model.")
            return cls()
'''

import os
os.makedirs('mental_health_chatbot/backend/models', exist_ok=True)
with open('mental_health_chatbot/backend/models/intent_classifier.py', 'w') as f:
    f.write(intent_code)

print("intent_classifier.py created!")

intent_classifier.py created!


In [52]:
import json
import pandas as pd
import os
import re

# Paths
BASE_PATH = "mental_health_chatbot/backend/data"
INPUT_FILE = os.path.join(BASE_PATH, "training_data.json")
CRISIS_FILE = os.path.join(BASE_PATH, "crisis_keywords.json")

# Load main dataset
try:
    with open(INPUT_FILE, "r") as f:
        data = json.load(f)
except json.JSONDecodeError:
    print(f"Warning: {INPUT_FILE} is empty or contains invalid JSON. Initializing with empty data.")
    data = []
except FileNotFoundError:
    print(f"Warning: {INPUT_FILE} not found. Initializing with empty data.")
    data = []

# Load crisis keywords dataset (VERY IMPORTANT)
try:
    with open(CRISIS_FILE, "r") as f:
        crisis_keywords = json.load(f)
except json.JSONDecodeError:
    print(f"Warning: {CRISIS_FILE} is empty or contains invalid JSON. Initializing with empty crisis keywords.")
    crisis_keywords = []
except FileNotFoundError:
    print(f"Warning: {CRISIS_FILE} not found. Initializing with empty crisis keywords.")
    crisis_keywords = []


print(f"Loaded {len(data)} main samples")
print(f"Loaded {len(crisis_keywords)} crisis keywords")

# ----------------------------
# TEXT CLEANING
# ----------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def classify_text(text):
    text = text.lower()

    if any(k in text for k in [
        "kill myself","suicide","end my life","want to die"
    ]):
        return "crisis"

    if any(k in text for k in [
        "help","advice","what should","how do","can you"
    ]):
        return "help_seeking"

    if any(k in text for k in ["hello","hi","thanks"]):
        return "general"

    return "sharing"

# ----------------------------
# PROCESS DATA
# ----------------------------
processed = []

for item in data:
    text = item.get("text", "").strip()
    if not text:
        continue

    clean = clean_text(text)
    label = classify_text(clean)

    processed.append({
        "text": clean,
        "intent": label
    })

# ----------------------------
# ADD EXTRA CRISIS DATA
# ----------------------------
for text in crisis_keywords:
    clean = clean_text(text)
    processed.append({
        "text": clean,
        "intent": "crisis"
    })

df = pd.DataFrame(processed)

# ----------------------------
# REMOVE DUPLICATES (VERY IMPORTANT)
# ----------------------------
df = df.drop_duplicates()
df = df[df['text'].str.len() > 5]

# ----------------------------
# BALANCE DATASET
# ----------------------------
min_count = df["intent"].value_counts().min()

df_balanced = pd.concat([
    group.sample(min_count, random_state=42)
    for _, group in df.groupby("intent")
]).reset_index(drop=True)

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# ----------------------------
# SAVE FILES
# ----------------------------
os.makedirs(BASE_PATH, exist_ok=True)

df_balanced.to_csv(os.path.join(BASE_PATH, "processed_intent_data.csv"), index=False)
df_balanced.to_json(os.path.join(BASE_PATH, "processed_intent_data.json"), orient="records", indent=2)

# ----------------------------
# OUTPUT
# ----------------------------
print("\nFINAL DATASET READY")
print(df_balanced["intent"].value_counts())
print(f"Total samples: {len(df_balanced)}")

Loaded 336 main samples
Loaded 5 crisis keywords

FINAL DATASET READY
intent
help_seeking    12
sharing         12
general         12
crisis          12
Name: count, dtype: int64
Total samples: 48


In [53]:
# Handle class imbalance using data augmentation

from sklearn.utils import resample
import pandas as pd
import os

# Load the processed data from the previous step
df = pd.read_csv('mental_health_chatbot/backend/data/processed_intent_data.csv')

# Separate by class, using 'intent' column
crisis_df = df[df['intent'] == 'crisis']
help_df = df[df['intent'] == 'help_seeking']
general_df = df[df['intent'] == 'general']
sharing_df = df[df['intent'] == 'sharing'] # Include sharing intent

print(f"Original class counts:")
print(f"Crisis: {len(crisis_df)}")
print(f"Help-seeking: {len(help_df)}")
print(f"General: {len(general_df)}")
print(f"Sharing: {len(sharing_df)}")

# Determine the target count for upsampling, considering only non-empty classes
all_dfs = [crisis_df, help_df, general_df, sharing_df]
existing_lengths = [len(d) for d in all_dfs if not d.empty]

if existing_lengths:
    target_count = max(existing_lengths)
else:
    # If all DataFrames are empty, target_count should be 0, resulting in an empty balanced_df
    target_count = 0

upsampled_dfs = []

for current_df in all_dfs:
    if not current_df.empty and target_count > 0:
        upsampled_df = resample(current_df,
                                replace=True,
                                n_samples=target_count,
                                random_state=42)
        upsampled_dfs.append(upsampled_df)
    else:
        # If a DataFrame is empty or target_count is 0, add the original (empty or non-upsampled) DF
        upsampled_dfs.append(current_df)
        if current_df.empty:
            print(f"Warning: A class DataFrame was empty, skipping upsampling for it.")

# Combine back together
if upsampled_dfs:
    balanced_df = pd.concat(upsampled_dfs)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
else:
    # If all input DataFrames were empty, balanced_df should also be empty
    balanced_df = pd.DataFrame(columns=df.columns)

print(f"\nBalanced class counts:")
if not balanced_df.empty:
    print(balanced_df['intent'].value_counts()) # Use 'intent' column for value counts
else:
    print("Balanced DataFrame is empty.")

# Save balanced data
os.makedirs('mental_health_chatbot/backend/data', exist_ok=True)
balanced_df.to_csv('mental_health_chatbot/backend/data/balanced_intent_data.csv', index=False)

Original class counts:
Crisis: 12
Help-seeking: 12
General: 12
Sharing: 12

Balanced class counts:
intent
general         12
sharing         12
help_seeking    12
crisis          12
Name: count, dtype: int64


In [54]:
# Split data into train and test sets

from sklearn.model_selection import train_test_split

# Split: 50% train, 50% test (adjusted for stratification with small dataset)
train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.2, # Increased test_size to ensure enough samples per class for stratification
    stratify=balanced_df['intent'],  # Maintain class distribution, using 'intent' column
    random_state=42
)

print(f"Dataset split:")
print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

print(f"\nTraining set distribution:")
print(train_df['intent'].value_counts())

print(f"\nTest set distribution:")
print(test_df['intent'].value_counts())

# Save splits
train_df.to_csv('mental_health_chatbot/backend/data/train_intent_data.csv', index=False)
test_df.to_csv('mental_health_chatbot/backend/data/test_intent_data.csv', index=False)

Dataset split:
Training samples: 38
Test samples: 10

Training set distribution:
intent
sharing         10
general         10
help_seeking     9
crisis           9
Name: count, dtype: int64

Test set distribution:
intent
help_seeking    3
crisis          3
general         2
sharing         2
Name: count, dtype: int64


In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

file_path = 'mental_health_chatbot/backend/data/train_intent_data.csv'

try:
    train_df = pd.read_csv(file_path)
    print("Loaded train_intent_data.csv from file.")
except FileNotFoundError:
    print(f"Error: {file_path} not found. Re-creating train_df from balanced_df.")

    # Ensure balanced_df is loaded if not already in scope
    # It is available in the kernel state, but explicitly loading it makes the cell more robust.
    balanced_df = pd.read_csv('mental_health_chatbot/backend/data/balanced_intent_data.csv')

    # Perform the train-test split as done in a later cell (sv1HamACFswA)
    train_df, test_df = train_test_split(
        balanced_df,
        test_size=0.2,
        stratify=balanced_df['intent'],  # Maintain class distribution
        random_state=42
    )
    # Save the created train_df (and test_df) to disk, just like in sv1HamACFswA
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    train_df.to_csv(file_path, index=False)
    test_df.to_csv('mental_health_chatbot/backend/data/test_intent_data.csv', index=False)
    print(f"Created and saved {file_path} and test_intent_data.csv.")

train_df = train_df.drop_duplicates(subset=['text'])

texts = train_df['text'].tolist()
labels = train_df['intent'].tolist()

print("\nTrain DataFrame label distribution:")
print(train_df['intent'].value_counts())

Loaded train_intent_data.csv from file.

Train DataFrame label distribution:
intent
crisis          7
sharing         6
help_seeking    5
general         5
Name: count, dtype: int64


In [56]:
# Entity Extractor Module

import spacy
from typing import List, Dict

class EntityExtractor:
    def __init__(self):
        """Initialize entity extractor for mental health entities"""
        print("Loading entity extraction model...")

        try:
            self.nlp = spacy.load("en_core_web_sm")
        except:
            import os
            os.system("python -m spacy download en_core_web_sm")
            self.nlp = spacy.load("en_core_web_sm")

        # Mental health entity dictionaries
        self.symptoms = {
            'insomnia', 'sleeplessness', 'nightmares', 'oversleeping', "can't sleep",
            'trouble sleeping', 'sadness', 'crying', 'mood swings', 'irritability',
            'fatigue', 'exhaustion', 'headache', 'panic attacks', 'heart racing',
            'suicidal thoughts', 'hopelessness', 'worthlessness', 'brain fog'
        }

        self.emotions = {
            'sad', 'depressed', 'anxious', 'worried', 'scared', 'angry', 'lonely',
            'hopeless', 'overwhelmed', 'numb', 'guilty', 'empty'
        }

        self.behaviors = {
            'isolation', 'withdrawing', 'self-harm', 'cutting', 'avoiding people',
            'staying in bed', 'not eating', 'drinking', 'using drugs'
        }

        print("Entity extractor initialized!")

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract mental health related entities from text"""
        text_lower = text.lower()
        extracted = {'symptoms': [], 'emotions': [], 'behaviors': []}

        # Enhanced symptom matching
        symptom_map = {
            "can't sleep": "insomnia",
            "cannot sleep": "insomnia",
            "trouble sleeping": "insomnia",
            "headache": "headache",
            "headaches": "headache",
            "racing thoughts": "overthinking",
            "heart racing": "heart racing",
            "feel empty": "empty",
            "feel numb": "numb",
            "no motivation": "low motivation",
            "no energy": "fatigue"
        }

        for original, mapped in symptom_map.items():
            if original in text_lower:
                extracted['symptoms'].append(mapped)

        # Standard matching
        for symptom in self.symptoms:
            if symptom in text_lower:
                extracted['symptoms'].append(symptom)

        for emotion in self.emotions:
            if emotion in text_lower or f"feel {emotion}" in text_lower or f"feeling {emotion}" in text_lower:
                extracted['emotions'].append(emotion)

        for behavior in self.behaviors:
            if behavior in text_lower:
                extracted['behaviors'].append(behavior)

        # Remove duplicates
        for key in extracted:
            extracted[key] = list(dict.fromkeys(extracted[key]))

        return extracted
# Save the entity extractor
entity_code = '''import spacy
from typing import List, Dict

class EntityExtractor:
    def __init__(self):
        """Initialize entity extractor for mental health entities"""
        print("Loading entity extraction model...")

        try:
            self.nlp = spacy.load("en_core_web_sm")
        except:
            import os
            os.system("python -m spacy download en_core_web_sm")
            self.nlp = spacy.load("en_core_web_sm")

        # Mental health entity dictionaries
        self.symptoms = {
            'insomnia', 'sleeplessness', 'nightmares', 'oversleeping', "can't sleep",
            'trouble sleeping', 'sadness', 'crying', 'mood swings', 'irritability',
            'fatigue', 'exhaustion', 'headache', 'panic attacks', 'heart racing',
            'suicidal thoughts', 'hopelessness', 'worthlessness', 'brain fog'
        }

        self.emotions = {
            'sad', 'depressed', 'anxious', 'worried', 'scared', 'angry', 'lonely',
            'hopeless', 'overwhelmed', 'numb', 'guilty', 'empty'
        }

        self.behaviors = {
            'isolation', 'withdrawing', 'self-harm', 'cutting', 'avoiding people',
            'staying in bed', 'not eating', 'drinking', 'using drugs'
        }

        print("Entity extractor initialized!")

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract mental health related entities from text"""
        text_lower = text.lower()
        extracted = {'symptoms': [], 'emotions': [], 'behaviors': []}

        # Enhanced symptom matching
        symptom_map = {
            "can't sleep": "insomnia",
            "cannot sleep": "insomnia",
            "trouble sleeping": "insomnia",
            "headache": "headache",
            "headaches": "headache",
            "racing thoughts": "overthinking",
            "heart racing": "heart racing",
            "feel empty": "empty",
            "feel numb": "numb",
            "no motivation": "low motivation",
            "no energy": "fatigue"
        }

        for original, mapped in symptom_map.items():
            if original in text_lower:
                extracted['symptoms'].append(mapped)

        # Standard matching
        for symptom in self.symptoms:
            if symptom in text_lower:
                extracted['symptoms'].append(symptom)

        for emotion in self.emotions:
            if emotion in text_lower or f"feel {emotion}" in text_lower or f"feeling {emotion}" in text_lower:
                extracted['emotions'].append(emotion)

        for behavior in self.behaviors:
            if behavior in text_lower:
                extracted['behaviors'].append(behavior)

        # Remove duplicates
        for key in extracted:
            extracted[key] = list(dict.fromkeys(extracted[key]))

        return extracted
'''

with open('mental_health_chatbot/backend/models/entity_extractor.py', 'w') as f:
    f.write(entity_code)

print("entity_extractor.py created!")

entity_extractor.py created!


In [76]:
import sys
import os
from typing import Dict, List

class MentalHealthAnalyzer:
    def __init__(self):
        print("Initializing Mental Health Analyzer...")

        from models.emotion_detector import EmotionDetector
        from models.intent_classifier import IntentClassifier
        from models.entity_extractor import EntityExtractor

        self.emotion_detector = EmotionDetector()
        self.intent_classifier = IntentClassifier.load(
            'mental_health_chatbot/backend/models/intent_classifier_trained'
        )
        self.entity_extractor = EntityExtractor()

        print("Mental Health Analyzer ready!")

    def analyze(self, text: str) -> dict:
        """Complete analysis of user input"""

        emotion_result = self.emotion_detector.detect(text)
        intent_result = self.intent_classifier.classify(text)
        entities = self.entity_extractor.extract_entities(text)

        analysis = {
            'text': text,
            'emotion': {
                'primary': emotion_result['top_emotion'],
                'confidence': emotion_result['confidence'],
                'all_emotions': emotion_result['all_emotions']
            },
            'intent': {
                'primary': intent_result['intent'],
                'confidence': intent_result['confidence'],
                'all_intents': intent_result['all_intents']
            },
            'entities': entities
        }

        analysis['is_crisis'] = self._check_crisis(text, intent_result, entities)
        analysis['risk_level'] = self._assess_risk_level(text, intent_result, entities)

        return analysis

    def _check_crisis(self, text: str, intent_result: dict, entities: dict) -> bool:
        """Check if the input indicates a crisis situation"""

        crisis_keywords = [
            'kill myself', 'suicide', 'end it all', 'want to die',
            'suicidal', 'overdose', 'jump off', 'end my life',
            'better off dead', 'not worth living'
        ]

        text_lower = text.lower()

        if intent_result['intent'] == 'crisis' and intent_result['confidence'] > 0.6:
            return True

        for keyword in crisis_keywords:
            if keyword in text_lower:
                return True

        if 'suicidal thoughts' in entities.get('symptoms', []):
            return True

        self_harm_behaviors = {'cutting', 'self-harm', 'hurting myself'}
        if any(behavior in entities.get('behaviors', []) for behavior in self_harm_behaviors):
            return True

        return False

    def _assess_risk_level(self, text: str, intent_result: dict, entities: dict) -> str:

        if self._check_crisis(text, intent_result, entities):
            return 'high'

        text_lower = text.lower()

        # Medium risk for physical symptoms of stress/anxiety
        physical_symptoms = ["headache", "muscle tension", "fatigue", "insomnia", "heart racing", "shortness of breath"]
        if any(symptom in text_lower for symptom in physical_symptoms):
            return 'medium'

        medium_indicators = [
            "depressed", "hopeless", "worthless", "cannot go on",
            "overwhelming", "unbearable", "giving up"
        ]

        if intent_result['intent'] == 'help_seeking':
            if any(symptom in entities.get('symptoms', []) for symptom in ['hopelessness', 'worthlessness']):
                return 'medium'
            if any(indicator in text_lower for indicator in medium_indicators):
                return 'medium'

        return 'low'

    def get_crisis_response(self) -> str:
        """Get appropriate crisis response message"""
        return """🚨 I'm very concerned about what you're sharing. Your safety is the top priority.

Please reach out for immediate help:

📞 Emergency Services (Police/Ambulance): **112**
📱 Tele MANAS (National Mental Health Helpline): **14416** or **1-800-91-4416** (24/7)
📞 AASRA: **022-27546669** (24/7)
📞 iCall (TISS): **022-25521111**

You don't have to face this alone. Help is available right now, and people care about you."""

# ==================== SAVE TO FILE ====================

with open('mental_health_chatbot/backend/models/mental_health_analyzer.py', 'w') as f:
    f.write("""import sys
import os
from typing import Dict, List

class MentalHealthAnalyzer:
    def __init__(self):
        print("Initializing Mental Health Analyzer...")

        from models.emotion_detector import EmotionDetector
        from models.intent_classifier import IntentClassifier
        from models.entity_extractor import EntityExtractor

        self.emotion_detector = EmotionDetector()
        self.intent_classifier = IntentClassifier.load(
            'mental_health_chatbot/backend/models/intent_classifier_trained'
        )
        self.entity_extractor = EntityExtractor()

        print("Mental Health Analyzer ready!")

    def analyze(self, text: str) -> dict:
        emotion_result = self.emotion_detector.detect(text)
        intent_result = self.intent_classifier.classify(text)
        entities = self.entity_extractor.extract_entities(text)

        analysis = {
            'text': text,
            'emotion': {
                'primary': emotion_result['top_emotion'],
                'confidence': emotion_result['confidence'],
                'all_emotions': emotion_result['all_emotions']
            },
            'intent': {
                'primary': intent_result['intent'],
                'confidence': intent_result['confidence'],
                'all_intents': intent_result['all_intents']
            },
            'entities': entities
        }

        analysis['is_crisis'] = self._check_crisis(text, intent_result, entities)
        analysis['risk_level'] = self._assess_risk_level(text, intent_result, entities)

        return analysis

    def _check_crisis(self, text: str, intent_result: dict, entities: dict) -> bool:
        crisis_keywords = [
            'kill myself', 'suicide', 'end it all', 'want to die',
            'suicidal', 'overdose', 'jump off', 'end my life',
            'better off dead', 'not worth living'
        ]

        text_lower = text.lower()

        if intent_result['intent'] == 'crisis' and intent_result['confidence'] > 0.6:
            return True

        for keyword in crisis_keywords:
            if keyword in text_lower:
                return True

        if 'suicidal thoughts' in entities.get('symptoms', []):
            return True

        self_harm_behaviors = {'cutting', 'self-harm', 'hurting myself'}
        if any(behavior in entities.get('behaviors', []) for behavior in self_harm_behaviors):
            return True

        return False

    def _assess_risk_level(self, text: str, intent_result: dict, entities: dict) -> str:

        if self._check_crisis(text, intent_result, entities):
            return 'high'

        text_lower = text.lower()

        # Medium risk for physical symptoms of stress/anxiety
        physical_symptoms = ["headache", "muscle tension", "fatigue", "insomnia", "heart racing", "shortness of breath"]
        if any(symptom in text_lower for symptom in physical_symptoms):
            return 'medium'

        medium_indicators = [
            "depressed", "hopeless", "worthless", "cannot go on",
            "overwhelming", "unbearable", "giving up"
        ]

        if intent_result['intent'] == 'help_seeking':
            if any(symptom in entities.get('symptoms', []) for symptom in ['hopelessness', 'worthlessness']):
                return 'medium'
            if any(indicator in text_lower for indicator in medium_indicators):
                return 'medium'

        return 'low'

    def get_crisis_response(self) -> str:
        return \"\"\"🚨 I'm very concerned about what you're sharing. Your safety is the top priority.

Please reach out for immediate help:

📞 Emergency Services (Police/Ambulance): **112**
📱 Tele MANAS (National Mental Health Helpline): **14416** or **1-800-91-4416** (24/7)
📞 AASRA: **022-27546669** (24/7)
📞 iCall (TISS): **022-25521111**

You don't have to face this alone. Help is available right now, and people care about you.\"\"\"""")


print("mental_health_analyzer.py created!")

mental_health_analyzer.py created!


In [77]:
# File: mental_health_chatbot/backend/utils/data_loader.py

data_utils_code = '''import json
import os
from typing import Dict, List, Optional

class DataLoader:
    """Utility class to load mental health data files"""

    def __init__(self, data_dir='mental_health_chatbot/backend/data'):
        self.data_dir = data_dir
        self.cbt_knowledge = None
        self.crisis_keywords = None
        self.mental_health_kg = None
        self.training_data = None

    def load_cbt_knowledge(self) -> List[Dict]:
        """Load CBT knowledge base"""
        path = os.path.join(self.data_dir, 'cbt_knowledge.json')
        if not os.path.exists(path):
            raise FileNotFoundError(f"cbt_knowledge.json not found at {path}")

        try:
            with open(path, 'r', encoding='utf-8') as f:
                self.cbt_knowledge = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Error decoding JSON from {path}: {e}. Please ensure the file is valid JSON.")

        print(f"Loaded {len(self.cbt_knowledge)} CBT techniques")
        return self.cbt_knowledge

    def load_crisis_keywords(self) -> Dict:
        """Load crisis detection keywords"""
        path = os.path.join(self.data_dir, 'crisis_keywords.json')
        if not os.path.exists(path):
            raise FileNotFoundError(f"crisis_keywords.json not found at {path}")

        try:
            with open(path, 'r', encoding='utf-8') as f:
                self.crisis_keywords = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Error decoding JSON from {path}: {e}. Please ensure the file is valid JSON.")

        print(f"Loaded crisis keywords")
        return self.crisis_keywords

    def load_knowledge_graph(self) -> Dict:
        """Load mental health knowledge graph"""
        path = os.path.join(self.data_dir, 'mental_health_kg.json')
        if not os.path.exists(path):
            raise FileNotFoundError(f"mental_health_kg.json not found at {path}")

        try:
            with open(path, 'r', encoding='utf-8') as f:
                self.mental_health_kg = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"Error decoding JSON from {path}: {e}. Please ensure the file is valid JSON.")

        print(f"Loaded knowledge graph with {len(self.mental_health_kg)} conditions")
        return self.mental_health_kg

    def check_crisis_keywords(self, text: str) -> Dict:
        """Check text for crisis keywords"""
        if not self.crisis_keywords:
            self.load_crisis_keywords()

        text_lower = text.lower()
        detected = {
            'is_crisis': False,
            'keywords_found': [],
            'risk_level': 'low',
            'categories': []
        }

        # Check suicide-related keywords
        for keyword in self.crisis_keywords['suicide_related']['direct_statements']:
            if keyword in text_lower:
                detected['is_crisis'] = True
                detected['keywords_found'].append(keyword)
                detected['categories'].append('suicide_related')

        # Check planning indicators
        for keyword in self.crisis_keywords['suicide_related']['planning_indicators']:
            if keyword in text_lower:
                detected['is_crisis'] = True
                detected['keywords_found'].append(keyword)
                detected['categories'].append('planning')
                detected['risk_level'] = 'high'

        return detected
'''

with open('mental_health_chatbot/backend/utils/data_loader.py', 'w') as f:
    f.write(data_utils_code)

print("data_loader.py created!")

data_loader.py created!


In [60]:
import sys
sys.modules.pop("models.intent_classifier", None)

In [61]:
# TEST: Intent Module

import sys
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import torch

# Force reload of the module to get latest changes
sys.modules.pop("models.intent_classifier", None)
sys.path.append('mental_health_chatbot/backend/models')

from intent_classifier import IntentClassifier

# ────────────────────────────────────────────
# STEP 1: Load & Merge All Data
# ────────────────────────────────────────────
DATA_DIR = 'mental_health_chatbot/backend/data'
all_records = []

print("Loading data files...\n")

# Load training_data.json
training_data_path = os.path.join(DATA_DIR, 'training_data.json')
try:
    with open(training_data_path, 'r', encoding='utf-8') as f:
        training_content = json.load(f)
    if isinstance(training_content, list):
        # Filter for items that actually have 'text' and 'intent'
        valid_training_data = [
            item for item in training_content
            if isinstance(item, dict) and 'text' in item and 'intent' in item
        ]
        all_records.extend(valid_training_data)
        print(f"  Loaded {len(valid_training_data)} records from training_data.json")
    else:
        print(f"  Skipped training_data.json: Expected a list of dictionaries, got {type(training_content)}")
except FileNotFoundError:
    print(f"  Warning: training_data.json not found at {training_data_path}")
except Exception as e:
    print(f"  Error loading training_data.json: {e}")


# Load crisis_keywords.json and extract crisis phrases
crisis_keywords_path = os.path.join(DATA_DIR, 'crisis_keywords.json')
try:
    with open(crisis_keywords_path, 'r', encoding='utf-8') as f:
        crisis_content = json.load(f)
    if isinstance(crisis_content, dict):
        crisis_phrases = []
        # Recursive function to extract all strings from nested dict/list
        def extract_strings(data_obj):
            if isinstance(data_obj, dict):
                for k, v in data_obj.items():
                    extract_strings(v)
            elif isinstance(data_obj, list):
                for item in data_obj:
                    extract_strings(item)
            elif isinstance(data_obj, str) and data_obj.strip():
                crisis_phrases.append(data_obj.strip())

        extract_strings(crisis_content)
        for phrase in crisis_phrases:
            all_records.append({'text': phrase, 'intent': 'crisis'})
        print(f"  Loaded {len(crisis_phrases)} crisis keywords from crisis_keywords.json")
    else:
        print(f"  Skipped crisis_keywords.json: Expected a dictionary, got {type(crisis_content)}")
except FileNotFoundError:
    print(f"  Warning: crisis_keywords.json not found at {crisis_keywords_path}")
except Exception as e:
    print(f"  Error loading crisis_keywords.json: {e}")

# Note: Other JSON files (cbt_knowledge.json, mental_health_kg.json/knowledge_graph.json)
# are not directly used for intent classification training, so they are not loaded here.

print(f"\nTotal raw records loaded for intent classification: {len(all_records)}")

# ────────────────────────────────────────────
# STEP 2: Build & Clean DataFrame
# ────────────────────────────────────────────
df = pd.DataFrame(all_records)

# Keep only valid rows
df = df[df['text'].notna() & df['intent'].notna()].copy()

# Clean text
df['text'] = (df['text']
              .str.lower()
              .str.replace(r'[^a-z0-9\s]', '', regex=True)
              .str.replace(r'\s+', ' ', regex=True)
              .str.strip())

df['intent'] = df['intent'].str.lower().str.strip()

# Filter valid intents
VALID_INTENTS = ['crisis', 'help_seeking', 'general']
df = df[df['intent'].isin(VALID_INTENTS)].copy()

# Remove duplicates and very short texts
df = df.drop_duplicates(subset=['text'])
df = df[df['text'].str.len() > 10]

print(f"\nAfter cleaning:")
print(df['intent'].value_counts())
print(f"Total usable samples: {len(df)}")

# ─────────────────────────────
# STEP 3: Balance Classes
# ─────────────────────────────
from sklearn.utils import resample

target_count = df['intent'].value_counts().max()
balanced_dfs = []

for intent in VALID_INTENTS:
    subset = df[df['intent'] == intent]
    if len(subset) == 0:
        continue
    upsampled = resample(subset, replace=True, n_samples=target_count, random_state=42)
    balanced_dfs.append(upsampled)

df_balanced = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter balancing:")
print(df_balanced['intent'].value_counts())
print(f"Total balanced samples: {len(df_balanced)}")

# ─────────────────────────────
# STEP 4: Train / Validation Split
# ─────────────────────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced['text'].tolist(),
    df_balanced['intent'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_balanced['intent']
)

print(f"\nTrain: {len(train_texts)} | Validation: {len(val_texts)}")
print(f"Train distribution: {pd.Series(train_labels).value_counts().to_dict()}")
print(f"Val distribution:   {pd.Series(val_labels).value_counts().to_dict()}")

# ─────────────────────────────
# STEP 5: Train Model
# ─────────────────────────────
clf = IntentClassifier(model_name="distilroberta-base", intents=VALID_INTENTS)

clf.train(
    train_texts=train_texts,
    train_labels=train_labels,
    eval_texts=val_texts,
    eval_labels=val_labels,
    epochs=5
)

# ─────────────────────────────
# STEP 6: Save Model
# ─────────────────────────────
SAVE_PATH = "mental_health_chatbot/backend/models/intent_classifier_trained"
clf.save(SAVE_PATH)

# ─────────────────────────────
# STEP 7: Final Evaluation
# ─────────────────────────────
print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)

val_results = clf.analyze_batch(val_texts)
y_true = val_labels
y_pred = [r['intent'] for r in val_results]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4, labels=VALID_INTENTS, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=VALID_INTENTS)
print(pd.DataFrame(cm, index=VALID_INTENTS, columns=VALID_INTENTS))

# Most important safety metric
crisis_recall = sum(1 for t, p in zip(y_true, y_pred) if t == 'crisis' and p == 'crisis') / sum(1 for t in y_true if t == 'crisis')
print(f"\n❂️ Crisis Recall: {crisis_recall:.4f} ← Critical Safety Metric")

if crisis_recall < 0.85:
    print("⚠️  Warning: Crisis recall is low. Consider adding more diverse crisis examples.")

# ─────────────────────────────
# STEP 8: Smoke Test
# ─────────────────────────────
print("\n✨ Smoke Test:")
test_cases = [
    "I want to end everything, I can't take this anymore",
    "Can you help me find a therapist? I'm struggling badly",
    "Today was okay, just wanted to chat a bit"
]

for text in test_cases:
    result = clf.classify(text)
    print(f"  [{result['intent'].upper():12}] ({result['confidence']:.1%}) → {text[:70]}")

Loading data files...

  Loaded 336 records from training_data.json
  Loaded 47 crisis keywords from crisis_keywords.json

Total raw records loaded for intent classification: 383

After cleaning:
intent
general         154
help_seeking    132
crisis           97
Name: count, dtype: int64
Total usable samples: 383

After balancing:
intent
help_seeking    154
crisis          154
general         154
Name: count, dtype: int64
Total balanced samples: 462

Train: 369 | Validation: 93
Train distribution: {'general': 123, 'crisis': 123, 'help_seeking': 123}
Val distribution:   {'crisis': 31, 'help_seeking': 31, 'general': 31}
Loading intent classification model...


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda (Tesla T4)
Initialized with 3 intents: ['crisis', 'help_seeking', 'general']
Starting training for IntentClassifier...


Map:   0%|          | 0/369 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training completed!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to mental_health_chatbot/backend/models/intent_classifier_trained

FINAL EVALUATION

Classification Report:
              precision    recall  f1-score   support

      crisis     1.0000    0.9677    0.9836        31
help_seeking     0.8857    1.0000    0.9394        31
     general     1.0000    0.9032    0.9492        31

    accuracy                         0.9570        93
   macro avg     0.9619    0.9570    0.9574        93
weighted avg     0.9619    0.9570    0.9574        93


Confusion Matrix:
              crisis  help_seeking  general
crisis            30             1        0
help_seeking       0            31        0
general            0             3       28

❂️ Crisis Recall: 0.9677 ← Critical Safety Metric

✨ Smoke Test:
  [CRISIS      ] (93.6%) → I want to end everything, I can't take this anymore
  [HELP_SEEKING] (94.4%) → Can you help me find a therapist? I'm struggling badly
  [GENERAL     ] (97.4%) → Today was okay, just wanted to chat a bit


In [78]:
# File: mental_health_chatbot/backend/data/knowledge_graph.json

import json

INPUT = "mental_health_chatbot/backend/data/mental_health_kg.json"
OUTPUT = "mental_health_chatbot/backend/data/knowledge_graph.json"

with open(INPUT, "r") as f:
    data = json.load(f)

entities = []
relations = []

def add_entity(name, type_):
    entities.append({
        "id": name.lower().replace(" ", "_"),
        "type": type_
    })

def add_relation(src, rel, tgt):
    relations.append({
        "source": src,
        "relation": rel,
        "target": tgt
    })

for condition, details in data.items():
    cond_id = condition.lower()

    add_entity(cond_id, "condition")

    for symptom in details.get("symptoms", []):
        sym_id = symptom.lower().replace(" ", "_")
        add_entity(sym_id, "symptom")
        add_relation(cond_id, "has_symptom", sym_id)

    for cause in details.get("causes", []):
        cause_id = cause.lower().replace(" ", "_")
        add_entity(cause_id, "cause")
        add_relation(cond_id, "has_cause", cause_id)

    for treatment in details.get("treatments", []):
        treat_id = treatment.lower().replace(" ", "_")
        add_entity(treat_id, "treatment")
        add_relation(cond_id, "treated_by", treat_id)

# Remove duplicates
entities = [dict(t) for t in {tuple(d.items()) for d in entities}]
relations = [dict(t) for t in {tuple(d.items()) for d in relations}]

# Save
with open(OUTPUT, "w") as f:
    json.dump({
        "entities": entities,
        "relations": relations
    }, f, indent=2)

print("Knowledge Graph built successfully!")
print(f"Entities: {len(entities)}")
print(f"Relations: {len(relations)}")

Knowledge Graph built successfully!
Entities: 225
Relations: 318


In [63]:
# File: mental_health_chatbot/backend/models/graph_transformer.py

import torch
import torch.nn as nn
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data

class GraphTransformer(nn.Module):
    def __init__(self, node_features, hidden_dim, num_heads, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerConv(
                in_channels=node_features if i == 0 else hidden_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                concat=False
            )
            for i in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, 3)  # low/medium/high risk

    def forward(self, x, edge_index):
        for layer in self.layers:
            x = layer(x, edge_index)
            x = torch.relu(x)
        return self.classifier(x)

# Save the file
with open('mental_health_chatbot/backend/models/graph_transformer.py', 'w') as f:
    f.write('''import torch
import torch.nn as nn
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data

class GraphTransformer(nn.Module):
    def __init__(self, node_features, hidden_dim, num_heads, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerConv(
                in_channels=node_features if i == 0 else hidden_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                concat=False
            )
            for i in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, 3)  # low/medium/high risk

    def forward(self, x, edge_index):
        for layer in self.layers:
            x = layer(x, edge_index)
            x = torch.relu(x)
        return self.classifier(x)''')

print("graph_transformer.py created!")

graph_transformer.py created!


In [64]:
import torch
from torch_geometric.data import Data

# Dummy node features + edges
def build_graph_tensor(nodes):

    if not nodes:
        return None

    # simple encoding
    x = torch.randn(len(nodes), 8)

    edge_index = torch.tensor([
        [i, (i+1) % len(nodes)] for i in range(len(nodes))
    ]).t().contiguous()

    return Data(x=x, edge_index=edge_index)

graph_model = GraphTransformer(node_features=8, hidden_dim=16, num_heads=2, num_layers=2)
graph_model.eval()

def predict_graph_risk(nodes):

    data = build_graph_tensor(nodes)

    if data is None:
        return "low"

    with torch.no_grad():
        out = graph_model(data.x, data.edge_index)

    score = out.mean().item()

    if score > 0.5:
        return "high"
    elif score > 0:
        return "medium"
    return "low"

In [65]:
# File: mental_health_chatbot/backend/utils/graph_reasoner.py

import json

with open("mental_health_chatbot/backend/data/knowledge_graph.json") as f:
    kg = json.load(f)

def infer_condition(user_text):
    user_text = user_text.lower()

    matches = {}

    for rel in kg["relations"]:
        if rel["relation"] == "has_symptom":
            if rel["target"].replace("_", " ") in user_text:
                cond = rel["source"]
                matches[cond] = matches.get(cond, 0) + 1

    if not matches:
        return None

    # Return the condition with the most matching symptoms
    return max(matches, key=matches.get)

# Save the file
with open('mental_health_chatbot/backend/utils/graph_reasoner.py', 'w') as f:
    f.write('''import json

with open("mental_health_chatbot/backend/data/knowledge_graph.json") as f:
    kg = json.load(f)

def infer_condition(user_text):
    user_text = user_text.lower()

    matches = {}

    for rel in kg["relations"]:
        if rel["relation"] == "has_symptom":
            if rel["target"].replace("_", " ") in user_text:
                cond = rel["source"]
                matches[cond] = matches.get(cond, 0) + 1

    if not matches:
        return None

    # Return the condition with the most matching symptoms
    return max(matches, key=matches.get)''')

print("graph_reasoner.py created!")

graph_reasoner.py created!


In [66]:
# KNOWLEDGE GRAPH INTEGRATION

import json
import networkx as nx

# Load the full knowledge graph
with open('mental_health_chatbot/backend/data/knowledge_graph.json', 'r') as f:
    kg_data = json.load(f)

G = nx.DiGraph()

# Build full graph
for entity in kg_data.get('entities', []):
    G.add_node(entity['id'], type=entity['type'])

for rel in kg_data.get('relations', []):
    G.add_edge(rel['source'], rel['target'], relation=rel['relation'])

print(f"Full Knowledge Graph Loaded: {len(G.nodes)} nodes, {len(G.edges)} edges")

def advanced_graph_reasoning(user_entities):
    """Final improved fuzzy matching for graph reasoning"""
    if not user_entities:
        return [], []

    expanded = set()
    risk_indicators = []

    # Normalize user entities
    user_set = {e.lower().strip().replace(" ", "_") for e in user_entities}

    for entity in user_set:
        # Exact match
        if entity in G:
            expanded.add(entity)
            for neighbor in G.successors(entity):
                expanded.add(neighbor)
                rel = G[entity][neighbor].get('relation', '')
                if rel in ['has_symptom', 'caused_by']:
                    risk_indicators.append(f"{entity} → {neighbor}")
            continue

        # Fuzzy match: substring or word overlap
        for node in list(G.nodes()):
            node_clean = node.replace("_", " ")
            if (entity in node_clean or
                node_clean in entity or
                any(word in node_clean for word in entity.split("_"))):

                expanded.add(node)
                for neighbor in G.successors(node):
                    expanded.add(neighbor)
                    rel = G[node][neighbor].get('relation', '')
                    if rel in ['has_symptom', 'caused_by']:
                        risk_indicators.append(f"{entity} → {neighbor}")

    return list(expanded), risk_indicators

print("Advanced Graph Reasoning ready!")

Full Knowledge Graph Loaded: 217 nodes, 316 edges
Advanced Graph Reasoning ready!


In [67]:
# GRAPH TRANSFORMER

import torch
import torch.nn as nn
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data

class GraphTransformer(nn.Module):
    def __init__(self, node_features=8, hidden_dim=32, num_heads=4, num_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerConv(in_channels=node_features if i == 0 else hidden_dim,
                           out_channels=hidden_dim, heads=num_heads, concat=False)
            for i in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, 3)  # 0=low, 1=medium, 2=high

    def forward(self, x, edge_index):
        for layer in self.layers:
            x = layer(x, edge_index)
            x = torch.relu(x)
        return self.classifier(x)

# Initialize the model
graph_model = GraphTransformer()
graph_model.eval()

def predict_graph_risk(nodes):
    """Use Graph Transformer for risk prediction"""
    if len(nodes) < 2:
        return "low"

    # Create graph from nodes
    x = torch.randn(len(nodes), 8)  # Node features
    edge_index = torch.tensor([[i, (i+1) % len(nodes)] for i in range(len(nodes))]).t()

    with torch.no_grad():
        out = graph_model(x, edge_index)
        risk_score = torch.softmax(out.mean(dim=0), dim=0)

    risk_map = {0: "low", 1: "medium", 2: "high"}
    return risk_map[risk_score.argmax().item()]

print("Graph Transformer Integrated & Ready")


Graph Transformer Integrated & Ready


In [ ]:
# File: mental_health_chatbot/backend/utils/risk_classifier.py

def classify_risk(text, intent):
    text = text.lower()

    # HIGH RISK (STRICT)
    high_risk_keywords = [
        "kill myself", "suicide", "end my life",
        "want to die", "hurt myself", "self harm"
    ]

    if any(k in text for k in high_risk_keywords):
        return "high"

    # MEDIUM RISK
    medium_risk_keywords = [
        "hopeless", "worthless", "no reason to live",
        "tired of life", "giving up"
    ]

    if any(k in text for k in medium_risk_keywords):
        return "medium"

    # IMPORTANT CHANGE
    # Do NOT blindly trust intent == crisis
    if intent == "crisis":
        return "medium"

    if intent == "help_seeking":
        return "medium"

    # LOW RISK
    return "low"

# Save the file
with open('mental_health_chatbot/backend/utils/risk_classifier.py', 'w') as f:
    f.write('''def classify_risk(text, intent):
    text = text.lower()

    # HIGH RISK (STRICT)
    high_risk_keywords = [
        "kill myself", "suicide", "end my life",
        "want to die", "hurt myself", "self harm"
    ]

    if any(k in text for k in high_risk_keywords):
        return "high"

    # MEDIUM RISK
    medium_risk_keywords = [
        "hopeless", "worthless", "no reason to live",
        "tired of life", "giving up"
    ]

    if any(k in text for k in medium_risk_keywords):
        return "medium"

    # IMPORTANT CHANGE
    # Do NOT blindly trust intent == crisis
    if intent == "crisis":
        return "medium"

    if intent == "help_seeking":
        return "medium"

    # LOW RISK
    return "low"''')

print("risk_classifier.py created!")

risk_classifier.py created!


In [68]:
# File: mental_health_chatbot/backend/utils/rag_module.py

import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class CBT_RAG:
    def __init__(self):
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.cbt_data = None
        self.index = None
        self.texts = []
        self.load_cbt_knowledge()

    def load_cbt_knowledge(self):
        with open('mental_health_chatbot/backend/data/cbt_knowledge.json', 'r') as f:
            self.cbt_data = json.load(f)

        self.texts = [item['technique'] + ": " + item['description'] for item in self.cbt_data]
        embeddings = self.embedder.encode(self.texts)
        embeddings = np.array(embeddings).astype('float32')

        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)
        print(f"CBT RAG Loaded: {len(self.cbt_data)} techniques")

    def retrieve(self, query, k=2):
        query_vec = self.embedder.encode([query]).astype('float32')
        D, I = self.index.search(query_vec, k)
        return [self.cbt_data[i] for i in I[0]]

# Initialize
cbt_rag = CBT_RAG()

# Save the file
with open('mental_health_chatbot/backend/utils/rag_module.py', 'w') as f:
    f.write('''import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class CBT_RAG:
    def __init__(self):
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.cbt_data = None
        self.index = None
        self.texts = []
        self.load_cbt_knowledge()

    def load_cbt_knowledge(self):
        with open('mental_health_chatbot/backend/data/cbt_knowledge.json', 'r') as f:
            self.cbt_data = json.load(f)

        self.texts = [item['technique'] + ": " + item['description'] for item in self.cbt_data]
        embeddings = self.embedder.encode(self.texts)
        embeddings = np.array(embeddings).astype('float32')

        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)
        print(f"CBT RAG Loaded: {len(self.cbt_data)} techniques")

    def retrieve(self, query, k=2):
        query_vec = self.embedder.encode([query]).astype('float32')
        D, I = self.index.search(query_vec, k)
        return [self.cbt_data[i] for i in I[0]]

# Initialize
cbt_rag = CBT_RAG()''')

print("rag_module.py created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CBT RAG Loaded: 50 techniques
rag_module.py created!


In [69]:
# # File: mental_health_chatbot/backend/models/llm_generator.py
import os
os.makedirs('mental_health_chatbot/backend/models', exist_ok=True)

llm_code = r'''from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

class LLMResponseGenerator:
    def __init__(self, model_name="Qwen/Qwen2-1.5B-Instruct"):
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        print("LLM Ready!")

    def generate(self, user_input, analysis, retrieved_cbt, risk_level):
        emotion = analysis['emotion']['primary']

        # Use Chat Template to force the model to behave correctly
        messages = [
            {"role": "system", "content": f"You are Elena, a supportive friend. Give a warm, direct 2-sentence response. User feels {emotion}."},
            {"role": "user", "content": user_input}
        ]

        # This converts the message list into the specific format Qwen expects
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            generated_ids = self.model.generate(
                model_inputs.input_ids,
                max_new_tokens=60,
                temperature=0.7,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )

        # Only take the part the model generated, not the prompt
        response_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = self.tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

        return response.strip()
'''

with open('mental_health_chatbot/backend/models/llm_generator.py', 'w', encoding='utf-8') as f:
    f.write(llm_code)

print("llm_generator.py created!")

llm_generator.py created!


In [79]:
import sys
import torch

# Clear memory to prevent crashes
torch.cuda.empty_cache()

# Correctly add the parent directory of 'models' and 'utils' to sys.path
sys.path.append('mental_health_chatbot/backend')

sys.modules.pop("models.llm_generator", None)

from models.mental_health_analyzer import MentalHealthAnalyzer
from models.llm_generator import LLMResponseGenerator
from utils.rag_module import cbt_rag # Corrected import path for rag_module
from utils.graph_reasoner import infer_condition # Added for completeness as it might be used later
from utils.risk_classifier import classify_risk # Added for completeness as it might be used later
from utils.data_loader import DataLoader # Added for completeness as it might be used later

# Re-initialize the analyzer with the corrected path for intent_classifier
analyzer = MentalHealthAnalyzer()
llm_generator = LLMResponseGenerator()

print("All core components loaded successfully!")

Initializing Mental Health Analyzer...
Loading emotion detection model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Emotion detector loaded!
Loading intent classifier from mental_health_chatbot/backend/models/intent_classifier_trained...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Intent classifier loaded successfully!
Loading entity extraction model...
Entity extractor initialized!
Mental Health Analyzer ready!
Loading Qwen/Qwen2-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM Ready!
All core components loaded successfully!


In [71]:
# File: mental_health_chatbot/backend/utils/explainer.py

import shap
import lime

class ExplainableAI:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def explain_risk(self, text, prediction):
        """Generate explanation for risk prediction"""

        # SHAP or LIME explanation
        explanation = {
            'risk_level': prediction,
            'key_indicators': self.extract_indicators(text),
            'reasoning': self.generate_reasoning(text, prediction),
            'attention_weights': self.get_attention(text)
        }
        return explanation

    def extract_indicators(self, text):
        """Extract crisis indicators from text"""
        indicators = []
        crisis_words = ['suicide', 'hopeless', 'worthless', 'kill']

        for word in crisis_words:
            if word in text.lower():
                indicators.append(word)

        return indicators

    def generate_reasoning(self, text, risk):
        """Generate human-readable explanation"""
        if risk == "high":
            return "High risk detected due to crisis keywords and negative emotional pattern"
        elif risk == "medium":
            return "Medium risk detected due to help-seeking intent and concerning symptoms"
        else:
            return "Low risk - general conversation or positive indicators"

# Save the file
with open('mental_health_chatbot/backend/utils/explainer.py', 'w') as f:
    f.write('''import shap
import lime

class ExplainableAI:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def explain_risk(self, text, prediction):
        """Generate explanation for risk prediction"""

        # SHAP or LIME explanation
        explanation = {
            'risk_level': prediction,
            'key_indicators': self.extract_indicators(text),
            'reasoning': self.generate_reasoning(text, prediction),
            'attention_weights': self.get_attention(text)
        }
        return explanation

    def extract_indicators(self, text):
        """Extract crisis indicators from text"""
        indicators = []
        crisis_words = ['suicide', 'hopeless', 'worthless', 'kill']

        for word in crisis_words:
            if word in text.lower():
                indicators.append(word)

        return indicators

    def generate_reasoning(self, text, risk):
        """Generate human-readable explanation"""
        if risk == "high":
            return "High risk detected due to crisis keywords and negative emotional pattern"
        elif risk == "medium":
            return "Medium risk detected due to help-seeking intent and concerning symptoms"
        else:
            return "Low risk - general conversation or positive indicators"''')

print("explainer.py created!")

explainer.py created!


In [72]:
# File: mental_health_chatbot/backend/utils/dynamic_graph_builder.py

import torch
from torch_geometric.data import Data

class DynamicGraphBuilder:
    def __init__(self):
        self.node_types = {
            'symptom': 0,
            'emotion': 1,
            'behavior': 2,
            'condition': 3
        }

    def build_user_graph(self, entities, base_kg):
        """Build personalized graph from user entities + base KG"""

        nodes = []
        edges = []

        # Add user's extracted entities as nodes
        for symptom in entities['symptoms']:
            nodes.append({'name': symptom, 'type': 'symptom'})

        for emotion in entities['emotions']:
            nodes.append({'name': emotion, 'type': 'emotion'})

        # Connect to base KG
        for node in nodes:
            related = self.find_related_in_kg(node['name'], base_kg)
            for rel in related:
                edges.append((node['name'], rel))

        # Convert to PyG format
        return self.to_pyg_data(nodes, edges)

    def to_pyg_data(self, nodes, edges):
        # Create node feature matrix
        x = torch.tensor([[self.node_types[n['type']]] for n in nodes])

        # Create edge index
        edge_index = torch.tensor(edges).t()

        return Data(x=x, edge_index=edge_index)

# Save the file
with open('mental_health_chatbot/backend/utils/dynamic_graph_builder.py', 'w') as f:
    f.write('''import torch
from torch_geometric.data import Data

class DynamicGraphBuilder:
    def __init__(self):
        self.node_types = {
            'symptom': 0,
            'emotion': 1,
            'behavior': 2,
            'condition': 3
        }

    def build_user_graph(self, entities, base_kg):
        """Build personalized graph from user entities + base KG"""

        nodes = []
        edges = []

        # Add user's extracted entities as nodes
        for symptom in entities['symptoms']:
            nodes.append({'name': symptom, 'type': 'symptom'})

        for emotion in entities['emotions']:
            nodes.append({'name': emotion, 'type': 'emotion'})

        # Connect to base KG
        for node in nodes:
            related = self.find_related_in_kg(node['name'], base_kg)
            for rel in related:
                edges.append((node['name'], rel))

        # Convert to PyG format
        return self.to_pyg_data(nodes, edges)

    def to_pyg_data(self, nodes, edges):
        # Create node feature matrix
        x = torch.tensor([[self.node_types[n['type']]] for n in nodes])

        # Create edge index
        edge_index = torch.tensor(edges).t()

        return Data(x=x, edge_index=edge_index)''')

print("dynamic_graph_builder.py created!")

dynamic_graph_builder.py created!


In [73]:
# XAI EXPLAINABILITY

def generate_explanation(analysis, risk_indicators, retrieved_cbt, final_risk):
    """Generate human-readable explanation"""
    explanation = f"""
Risk Level: {final_risk.upper()}
Primary Emotion: {analysis['emotion']['primary']}
Key Symptoms Detected: {', '.join(analysis['entities']['symptoms'][:4]) or 'None'}
Graph Reasoning: {risk_indicators[:3] or 'No strong connections found'}
Knowledge Used: {retrieved_cbt['technique'] if retrieved_cbt else 'General support'}
Confidence: {round(analysis['intent']['confidence']*100, 1)}%
"""
    return explanation.strip()

print("XAI Explainability")

XAI Explainability


In [ ]:
# =============================================================================
# 💜 Elena — Professional Mental Health Companion UI
# =============================================================================

import sys
import os
import time
import gradio as gr

# Add paths
sys.path.append('/content/mental_health_chatbot/backend')
sys.path.append('/content/mental_health_chatbot/backend/models')
sys.path.append('/content/mental_health_chatbot/backend/utils')

from models.mental_health_analyzer import MentalHealthAnalyzer

print("🚀 Loading Elena...")

analyzer = MentalHealthAnalyzer()

print("✅ Elena is ready.")

BOT_NAME = "Elena"

# -----------------------------
# MEMORY
# -----------------------------
conversation_memory = []

def update_memory(user, bot):
    conversation_memory.append({"user": user, "bot": bot})

# -----------------------------
# TYPING EFFECT
# -----------------------------
def typing_effect(text):
    output = ""
    for char in text:
        output += char
        yield output
        time.sleep(0.012)

# ==============================
# LATEST CHATBOT RESPONSE FUNCTION (Updated Logic)
# ==============================
def chatbot_response(user_input):
    try:
        # 1. Core Analysis
        analysis = analyzer.analyze(user_input)
        entities = analysis['entities']
        graph_nodes = list(entities['symptoms']) + list(entities['emotions'])

        # 2. Graph Reasoning (Always run)
        expanded, risk_indicators = advanced_graph_reasoning(graph_nodes)

        # 3. Graph Transformer Risk Assessment (Core Module)
        transformer_risk = predict_graph_risk(graph_nodes)

        # 4. RAG (CBT)
        retrieved_cbt = cbt_rag.retrieve(user_input, k=1)[0] if 'cbt_rag' in globals() else None

        # 5. Final Risk (Combined from Analyzer + Graph Transformer)
        final_risk = analysis['risk_level']
        if transformer_risk == "high" or len(risk_indicators) > 2:
            final_risk = "high"
        elif transformer_risk == "medium" or len(risk_indicators) > 0:
            final_risk = "medium"

        # 6. Generate Response
        if final_risk == "high":
            response_text = analyzer.get_crisis_response()
        else:
            response_text = llm_generator.generate(user_input, analysis, retrieved_cbt, final_risk)

        # 7. XAI Explanation (Graph Reasoning always shown)
        explanation = f"""
Risk Level: {final_risk.upper()}
Primary Emotion: {analysis['emotion']['primary']}
Key Symptoms: {', '.join(entities['symptoms'][:3]) or 'None'}
Graph Connections: {', '.join(risk_indicators[:3]) or 'No strong connections found'}
Graph Transformer Risk: {transformer_risk.upper()}
Knowledge Used: {retrieved_cbt['technique'] if retrieved_cbt else 'General support'}
"""

        return {
            "risk": final_risk,
            "response": response_text,
            "emotion": analysis['emotion']['primary'],
            "explanation": explanation.strip()
        }

    except Exception as e:
        print(f"Error: {e}")
        return {
            "risk": "low",
            "response": "I'm here to listen. Tell me more about how you're feeling.",
            "emotion": "neutral",
            "explanation": "System used fallback mode."
        }

# -----------------------------
# MAIN CHAT FUNCTION
# -----------------------------
def chatbot_fn(message, history):
    if not message or not message.strip():
        return history, ""

    try:
        result = chatbot_response(message)

        final_response = result["response"]
        explanation = result.get("explanation", "")

        if explanation:
            final_response += f"\n\n🧠 **Reasoning**: {explanation}"

        update_memory(message, final_response)

    except Exception as e:
        print(f"FULL ERROR: {type(e).__name__}: {e}")
        final_response = "I'm having trouble responding right now. Please try again in a moment."

    history.append((message, ""))

    for partial in typing_effect(final_response):
        history[-1] = (message, partial)
        yield history, ""

    return history, ""

# -----------------------------
# PROFESSIONAL GRADIO UI
# -----------------------------
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="purple",
        neutral_hue="slate"
    ),
    title="Elena — Mental Health Companion",
    css="""
    .gradio-container {max-width: 900px; margin: auto;}
    .chatbot {height: 680px !important;}
    .message {border-radius: 12px;}
    """
) as demo:

    gr.Markdown("""
    # 💜 Elena — Your Mental Health Companion

    A safe, compassionate, and intelligent space to talk.
    *Your privacy and wellbeing are our top priority.*
    """)

    chatbot = gr.Chatbot(
        height=680,
        show_label=False,
        bubble_full_width=False
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Share what's on your mind... I'm here to listen without judgment.",
            lines=3,
            scale=5,
            container=False,
            show_label=False
        )
        send = gr.Button("Send", variant="primary", scale=1)

    with gr.Row():
        clear = gr.Button("New Conversation", variant="secondary")
        gr.Markdown(
            """<p style='text-align: center; color: #666; font-size: 0.9em;'>
            Elena is not a substitute for professional mental health care.
            In crisis, please call 988 or local emergency services.
            </p>""",
            visible=True
        )

    # Interactions
    msg.submit(chatbot_fn, [msg, chatbot], [chatbot, msg])
    send.click(chatbot_fn, [msg, chatbot], [chatbot, msg])
    clear.click(lambda: [], None, chatbot, queue=False)

    # Footer
    gr.Markdown("---")
    gr.Markdown(
        "**Confidential & Supportive Space** • Built with care for mental wellbeing"
    )

demo.launch(
    share=True,
    debug=True,
    show_error=True
)

🚀 Loading Elena...
Initializing Mental Health Analyzer...
Loading emotion detection model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Emotion detector loaded!
Loading intent classifier from mental_health_chatbot/backend/models/intent_classifier_trained...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Intent classifier loaded successfully!
Loading entity extraction model...
Entity extractor initialized!
Mental Health Analyzer ready!
✅ Elena is ready.


/tmp/ipykernel_987/3600078772.py:134: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_987/3600078772.py:134: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_987/3600078772.py:155: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_987/3600078772.py:155: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_987/3600

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb5d278175dc887037.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
